In [6]:
!pip install seaborn 


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt


from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report, recall_score, precision_score, f1_score, confusion_matrix    
from sklearn.ensemble import RandomForestClassifier

In [5]:
df = pd.read_csv("paddydataset.csv")

print(df.shape)
print(df.head())

(2789, 45)
   Hectares      Agriblock      Variety Soil Types  Seedrate(in Kg)  \
0          6     Cuddalore        CO_43   alluvial              150   
1          6   Kurinjipadi      ponmani       clay              150   
2          6       Panruti  delux ponni   alluvial              150   
3          6  Kallakurichi        CO_43       clay              150   
4          6  Sankarapuram      ponmani   alluvial              150   

   LP_Mainfield(in Tonnes) Nursery  Nursery area (Cents)  \
0                     75.0     dry                   120   
1                     75.0     wet                   120   
2                     75.0     dry                   120   
3                     75.0     wet                   120   
4                     75.0     dry                   120   

   LP_nurseryarea(in Tonnes)  DAP_20days  ...  Wind Direction_D1_D30  \
0                          6         240  ...                     SW   
1                          6         240  ...            

In [8]:
df.columns = df.columns.str.strip()

In [9]:
print(df.isnull().sum())

Hectares                              0
Agriblock                             0
Variety                               0
Soil Types                            0
Seedrate(in Kg)                       0
LP_Mainfield(in Tonnes)               0
Nursery                               0
Nursery area (Cents)                  0
LP_nurseryarea(in Tonnes)             0
DAP_20days                            0
Weed28D_thiobencarb                   0
Urea_40Days                           0
Potassh_50Days                        0
Micronutrients_70Days                 0
Pest_60Day(in ml)                     0
30DRain( in mm)                       0
30DAI(in mm)                          0
30_50DRain( in mm)                    0
30_50DAI(in mm)                       0
51_70DRain(in mm)                     0
51_70AI(in mm)                        0
71_105DRain(in mm)                    0
71_105DAI(in mm)                      0
Min temp_D1_D30                       0
Max temp_D1_D30                       0


# Step 2: prepare data for classification

In [14]:
median = df['Paddy yield(in Kg)'].median()
print(median)

# create a classification target based on the median
# 1 = high yield, 0 = low yield
df['Yield_Class'] = (df['Paddy yield(in Kg)'] >= median).astype(int)
print(df['Yield_Class'].value_counts())
print('=='*50)
print(df['Yield_Class'].value_counts(normalize=True)*100)

24636.0
Yield_Class
1    1397
0    1392
Name: count, dtype: int64
Yield_Class
1    50.089638
0    49.910362
Name: proportion, dtype: float64


## Feature: X = maximum temperature(our pedictor)
## Target: y = high or low paddy yield(what we predicted)

In [15]:
X = df[['Max temp_D1_D30']]
print(X.head())
y = df['Yield_Class']
print(y.head())

   Max temp_D1_D30
0               34
1               34
2               35
3               33
4               32
0    1
1    1
2    1
3    1
4    1
Name: Yield_Class, dtype: int64


## Step 3: spliting into training and testing

In [24]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print(X_train)
print(y_train)
print(X_test)
print(y_test)



      Max temp_D1_D30
2699               34
344                34
839                33
1106               32
1747               33
...               ...
1638               35
1095               31
1130               32
1294               32
860                34

[1952 rows x 1 columns]
2699    0
344     1
839     1
1106    0
1747    0
       ..
1638    0
1095    1
1130    1
1294    1
860     1
Name: Yield_Class, Length: 1952, dtype: int64
      Max temp_D1_D30
1080               35
1517               33
1467               34
1029               31
2120               34
...               ...
1938               34
1034               32
1591               32
1508               34
2785               32

[837 rows x 1 columns]
1080    1
1517    0
1467    1
1029    1
2120    0
       ..
1938    0
1034    1
1591    0
1508    0
2785    0
Name: Yield_Class, Length: 837, dtype: int64


## Step 4: train logistic regression model

In [25]:
# Step 5: Train the Logistic Regression Model
print("=" * 60)
print("TRAINING LOGISTIC REGRESSION MODEL")
print("=" * 60)

model = LogisticRegression(random_state=42)
model.fit(X_train, y_train)

print("✓ Model Training Complete!\n")

# Extract learned parameters
intercept = model.intercept_[0]
coefficient = model.coef_[0][0]

print(f"Model Parameters Learned:")
print(f"  • Intercept (b₀): {intercept:.4f}")
print(f"  • Coefficient (b₁): {coefficient:.4f}")
print(f"\nInterpretation:")
print(f"  • For every 1°C increase in temperature,")
print(f"  • The log-odds of HIGH yield {'increases' if coefficient > 0 else 'decreases'} by {abs(coefficient):.4f}")

# Make predictions on training data
y_pred_train = model.predict(X_train)
y_proba_train = model.predict_proba(X_train)

print(f"\nExample Predictions (First 5 training samples):")
print(f"{'Temperature': >15} | {'Probability':<20} | {'Prediction':<12} | {'Actual':<8}")
print("-" * 65)
for i in range(5):
    temp = X_train.iloc[i, 0]
    proba = y_proba_train[i]
    pred = y_pred_train[i]
    actual = y_train.iloc[i]
    print(f"{temp:>15.2f}°C | P(Low)={proba[0]:.3f}, P(High)={proba[1]:.3f} | {'HIGH' if pred == 1 else 'LOW':<12} | {'HIGH' if actual == 1 else 'LOW':<8}")

TRAINING LOGISTIC REGRESSION MODEL
✓ Model Training Complete!

Model Parameters Learned:
  • Intercept (b₀): 0.5478
  • Coefficient (b₁): -0.0163

Interpretation:
  • For every 1°C increase in temperature,
  • The log-odds of HIGH yield decreases by 0.0163

Example Predictions (First 5 training samples):
    Temperature | Probability          | Prediction   | Actual  
-----------------------------------------------------------------
          34.00°C | P(Low)=0.502, P(High)=0.498 | LOW          | LOW     
          34.00°C | P(Low)=0.502, P(High)=0.498 | LOW          | HIGH    
          33.00°C | P(Low)=0.497, P(High)=0.503 | HIGH         | HIGH    
          32.00°C | P(Low)=0.493, P(High)=0.507 | HIGH         | LOW     
          33.00°C | P(Low)=0.497, P(High)=0.503 | HIGH         | LOW     


In [26]:
# Step 6: Make Predictions on Test Data
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)

print("=" * 60)
print("TEST SET PREDICTIONS (First 10 samples)")
print("=" * 60)
print(f"{'Temperature': >15} | {'P(High)':<10} | {'Prediction':<12} | {'Actual':<8} | {'Correct?':<8}")
print("-" * 65)
for i in range(min(10, len(X_test))):
    temp = X_test.iloc[i, 0]
    proba = y_proba[i, 1]
    pred = y_pred[i]
    actual = y_test.iloc[i]
    correct = "✓ YES" if pred == actual else "✗ NO"
    print(f"{temp:>15.2f}°C | {proba:<10.3f} | {'HIGH' if pred == 1 else 'LOW':<12} | {'HIGH' if actual == 1 else 'LOW':<8} | {correct:<8}")

TEST SET PREDICTIONS (First 10 samples)
    Temperature | P(High)    | Prediction   | Actual   | Correct?
-----------------------------------------------------------------
          35.00°C | 0.494      | LOW          | HIGH     | ✗ NO    
          33.00°C | 0.503      | HIGH         | LOW      | ✗ NO    
          34.00°C | 0.498      | LOW          | HIGH     | ✗ NO    
          31.00°C | 0.511      | HIGH         | HIGH     | ✓ YES   
          34.00°C | 0.498      | LOW          | LOW      | ✓ YES   
          33.00°C | 0.503      | HIGH         | HIGH     | ✓ YES   
          34.00°C | 0.498      | LOW          | LOW      | ✓ YES   
          32.00°C | 0.507      | HIGH         | HIGH     | ✓ YES   
          32.00°C | 0.507      | HIGH         | LOW      | ✗ NO    
          32.00°C | 0.507      | HIGH         | LOW      | ✗ NO    


---

##  Part 5: Model Analysis and Evaluation

### Understanding Evaluation Metrics

Before we see the results, let's understand what each metric means:

#### 1️⃣ **Accuracy**
- **Definition:** What percentage of predictions are correct?
- **Formula:** (Correct Predictions) / (Total Predictions) × 100
- **Example:** If we make 100 predictions and 85 are correct → 85% Accuracy
- **Range:** 0-100% (higher is better)

#### 2️⃣ **Precision**
- **Definition:** Of the cases we predicted as HIGH, how many actually were HIGH?
- **Real-world example:** Email spam filter → "Of emails marked SPAM, how many actually are spam?"
- **Formula:** True Positives / (True Positives + False Positives)
- **Why it matters:** Want to avoid false alarms

#### 3️⃣ **Recall (Sensitivity)**
- **Definition:** Of all actual HIGH cases, how many did we catch?
- **Real-world example:** Medical test → "Of all people with disease, how many did test catch?"
- **Formula:** True Positives / (True Positives + False Negatives)
- **Why it matters:** Don't want to miss real cases

#### 4️⃣ **F1-Score**
- **Definition:** Balanced score between Precision and Recall
- **When to use:** When you want both precision and recall to be good
- **Range:** 0-1 (higher is better)

#### 5️⃣ **Confusion Matrix**
```
                    PREDICTED
                 HIGH    LOW
ACTUAL  HIGH  |  TP  |  FN  |
        LOW   |  FP  |  TN  |

TP = True Positive (correctly predicted HIGH)
FN = False Negative (predicted LOW, actually HIGH)
FP = False Positive (predicted HIGH, actually LOW)
TN = True Negative (correctly predicted LOW)
```

In [28]:
# Step 7: Calculate Evaluation Metrics
print("\n" + "=" * 60)
print("MODEL EVALUATION METRICS")
print("=" * 60)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"\n1. ACCURACY: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"   → {int(accuracy * len(y_test))} out of {len(y_test)} predictions correct")

# Precision
precision = precision_score(y_test, y_pred)
print(f"\n2. PRECISION: {precision:.4f}")
print(f"   → Of cases predicted as HIGH, {precision*100:.2f}% actually were HIGH")
print(f"   → Fewer false alarms")

# Recall
recall = recall_score(y_test, y_pred)
print(f"\n3. RECALL: {recall:.4f}")
print(f"   → Of all actual HIGH cases, we caught {recall*100:.2f}%")
print(f"   → Fewer missed cases")

# F1-Score
f1 = f1_score(y_test, y_pred)
print(f"\n4. F1-SCORE: {f1:.4f}")
print(f"   → Balanced score between Precision and Recall")

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print(f"\n5. CONFUSION MATRIX:")
print(f"\n                    PREDICTED")
print(f"                   HIGH    LOW")
print(f"ACTUAL  HIGH    [{cm[1,1]:>3}]   [{cm[1,0]:>3}]")
print(f"        LOW     [{cm[0,1]:>3}]   [{cm[0,0]:>3}]")

tn, fp, fn, tp = cm.ravel()
print(f"\n   Details:")
print(f"   • True Positives (TP):   {tp} (correctly predicted HIGH)")
print(f"   • True Negatives (TN):   {tn} (correctly predicted LOW)")
print(f"   • False Positives (FP):  {fp} (incorrectly predicted HIGH)")
print(f"   • False Negatives (FN):  {fn} (incorrectly predicted LOW - MISSED)")

# Detailed Classification Report
print("\n" + "=" * 60)
print("DETAILED CLASSIFICATION REPORT")
print("=" * 60)
print(classification_report(y_test, y_pred, target_names=['Low Yield', 'High Yield']))


MODEL EVALUATION METRICS

1. ACCURACY: 0.4934 (49.34%)
   → 413 out of 837 predictions correct

2. PRECISION: 0.4916
   → Of cases predicted as HIGH, 49.16% actually were HIGH
   → Fewer false alarms

3. RECALL: 0.4940
   → Of all actual HIGH cases, we caught 49.40%
   → Fewer missed cases

4. F1-SCORE: 0.4928
   → Balanced score between Precision and Recall

5. CONFUSION MATRIX:

                    PREDICTED
                   HIGH    LOW
ACTUAL  HIGH    [206]   [211]
        LOW     [213]   [207]

   Details:
   • True Positives (TP):   206 (correctly predicted HIGH)
   • True Negatives (TN):   207 (correctly predicted LOW)
   • False Positives (FP):  213 (incorrectly predicted HIGH)
   • False Negatives (FN):  211 (incorrectly predicted LOW - MISSED)

DETAILED CLASSIFICATION REPORT
              precision    recall  f1-score   support

   Low Yield       0.50      0.49      0.49       420
  High Yield       0.49      0.49      0.49       417

    accuracy                          